# Solutions — Notebook 04: Lakehouse

**BINFX410 — Chapter 10**

> **Prerequisite:** Run `01_introduction_and_setup.ipynb` to generate `../raw_data/`. The setup cell below rebuilds the samples Delta table at a dedicated path so that time travel works regardless of whether VACUUM was already run in `04_lakehouse.ipynb`.

In [ ]:
import os
import shutil
import pandas as pd
import pyarrow as pa
import duckdb
from deltalake import DeltaTable, write_deltalake

# Use a dedicated path so VACUUM in the main notebook can't break time travel here
SOL_SAMPLES_PATH = '../lakehouse/sol_samples'
LAKEHOUSE_ROOT   = '../lakehouse'
CALLS_PATH       = f'{LAKEHOUSE_ROOT}/variant_calls'

# Rebuild fresh — three versions that mirror what 04_lakehouse.ipynb does:
#   v0: initial write of 500 samples
#   v1: append of 3 new 2025 samples
if os.path.exists(SOL_SAMPLES_PATH):
    shutil.rmtree(SOL_SAMPLES_PATH)

df_samples = pd.read_csv('../raw_data/samples.csv')
df_samples['collection_date'] = pd.to_datetime(df_samples['collection_date'])

# v0 — initial write
write_deltalake(SOL_SAMPLES_PATH, df_samples, mode='overwrite')

# v1 — append 3 new samples (same as main notebook section 5)
new_samples = pd.DataFrame([
    {'sample_id': 501, 'patient_id': 301, 'tissue_type': 'Tumor',
     'sequencing_platform': 'Illumina_NovaSeq', 'library_prep': 'WGS',
     'collection_date': pd.Timestamp('2025-01-15'),
     'mean_coverage': 95.3, 'pct_bases_20x': 0.9421, 'project_id': 'PROJ_001'},
    {'sample_id': 502, 'patient_id': 302, 'tissue_type': 'Blood',
     'sequencing_platform': 'Illumina_HiSeq', 'library_prep': 'WES',
     'collection_date': pd.Timestamp('2025-02-03'),
     'mean_coverage': 58.1, 'pct_bases_20x': 0.8834, 'project_id': 'PROJ_003'},
    {'sample_id': 503, 'patient_id': 303, 'tissue_type': 'Normal',
     'sequencing_platform': 'Oxford_Nanopore', 'library_prep': 'WGS',
     'collection_date': pd.Timestamp('2025-02-20'),
     'mean_coverage': 42.7, 'pct_bases_20x': 0.8190, 'project_id': 'PROJ_005'},
])
write_deltalake(SOL_SAMPLES_PATH, new_samples, mode='append')

dt = DeltaTable(SOL_SAMPLES_PATH)
print(f'sol_samples rebuilt: v{dt.version()}, {len(dt.to_pandas()):,} rows, {dt.get_add_actions().num_rows} files')

## Exercise 4.1 — Time Travel Audit

The samples table has been through several versions (initial write, append of 3 new samples).

a) Use `DeltaTable(SAMPLES_PATH, version=N).to_pandas()` to count rows at each version (0, 1, 2, ..., current).

b) Find samples whose `tissue_type` changed between version 0 and the current version. Hint: merge both DataFrames on `sample_id` and compare the `tissue_type` column.

In [ ]:
# a) Row counts at each version
dt_samples = DeltaTable(SOL_SAMPLES_PATH)
current_version = dt_samples.version()

print('=== Row counts at each version ===')
for v in range(current_version + 1):
    df_v = DeltaTable(SOL_SAMPLES_PATH, version=v).to_pandas()
    print(f'  v{v}: {len(df_v):,} rows')

# b) Records where tissue_type changed between v0 and current
df_v0      = DeltaTable(SOL_SAMPLES_PATH, version=0).to_pandas()
df_current = DeltaTable(SOL_SAMPLES_PATH).to_pandas()

merged = df_v0[['sample_id', 'tissue_type']].merge(
    df_current[['sample_id', 'tissue_type']],
    on='sample_id',
    suffixes=('_v0', '_current')
)
changed = merged[merged['tissue_type_v0'] != merged['tissue_type_current']]
print(f'\nSamples with changed tissue_type: {len(changed)}')
if len(changed):
    print(changed)
else:
    print('  (none — tissue_type was not changed in any merge operation)')

## Exercise 4.2 — Bulk MERGE on Variant Calls

Write a MERGE operation on the `variant_calls` table that:
- Updates the `filter_status` from `'LowQuality'` to `'Reanalysis_Needed'` for all calls where `call_date` is before 2022-01-01 (these old calls need to be re-run with the new pipeline)
- Does NOT insert new rows (only update existing)

Hint: Create a DataFrame of affected calls with the updated `filter_status`, then use `.when_matched_update()`.

In [ ]:
dt_calls_ex = DeltaTable(CALLS_PATH)
df_calls_now = dt_calls_ex.to_pandas()

# Step 1: Find LowQuality calls before 2022
df_calls_now['call_date'] = pd.to_datetime(df_calls_now['call_date'])
cutoff = pd.Timestamp('2022-01-01')
to_reanalyze = df_calls_now[
    (df_calls_now['filter_status'] == 'LowQuality') &
    (df_calls_now['call_date'] < cutoff)
].copy()
to_reanalyze['filter_status'] = 'Reanalysis_Needed'

print(f'Calls to update: {len(to_reanalyze)}')
print(to_reanalyze[['call_id', 'call_date', 'filter_status']].head())

if len(to_reanalyze) > 0:
    updates_arrow = pa.Table.from_pandas(to_reanalyze)

    # Step 2: MERGE — update only, no inserts
    (
        dt_calls_ex.merge(
            source=updates_arrow,
            predicate='target.call_id = source.call_id',
            source_alias='source',
            target_alias='target',
        )
        .when_matched_update({'filter_status': 'source.filter_status'})
        .execute()
    )

    dt_calls_ex = DeltaTable(CALLS_PATH)
    df_after = dt_calls_ex.to_pandas()
    print(f'\nAfter MERGE — version: {dt_calls_ex.version()}')
    print('filter_status counts:')
    print(df_after['filter_status'].value_counts())

## Exercise 4.3 — Cohort Retention via Delta Tables

Using DuckDB's `delta_scan()`, write a query that produces a **monthly cohort retention table** for samples: for each collection cohort (year+month), what percentage of samples had variant calls in month 0, 1, 2, and 3 after collection?

Expected output columns: `cohort_year`, `cohort_month`, `cohort_size`, `month_0_pct`, `month_1_pct`, `month_2_pct`, `month_3_pct`.

In [ ]:
con = duckdb.connect()
con.execute('INSTALL delta; LOAD delta;')

retention = con.execute("""
    WITH cohorts AS (
        SELECT
            s.sample_id,
            YEAR(s.collection_date)  AS cohort_year,
            MONTH(s.collection_date) AS cohort_month,
            s.collection_date
        FROM delta_scan('../lakehouse/samples') s
    ),
    call_activity AS (
        SELECT
            vc.sample_id,
            vc.call_date
        FROM delta_scan('../lakehouse/variant_calls') vc
        WHERE vc.filter_status = 'PASS'
    )
    SELECT
        c.cohort_year,
        c.cohort_month,
        COUNT(DISTINCT c.sample_id)                                     AS cohort_size,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN DATEDIFF('month', c.collection_date, ca.call_date) = 0
            THEN c.sample_id END) / COUNT(DISTINCT c.sample_id), 1)     AS month_0_pct,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN DATEDIFF('month', c.collection_date, ca.call_date) = 1
            THEN c.sample_id END) / COUNT(DISTINCT c.sample_id), 1)     AS month_1_pct,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN DATEDIFF('month', c.collection_date, ca.call_date) = 2
            THEN c.sample_id END) / COUNT(DISTINCT c.sample_id), 1)     AS month_2_pct,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN DATEDIFF('month', c.collection_date, ca.call_date) = 3
            THEN c.sample_id END) / COUNT(DISTINCT c.sample_id), 1)     AS month_3_pct
    FROM cohorts c
    LEFT JOIN call_activity ca ON c.sample_id = ca.sample_id
    GROUP BY c.cohort_year, c.cohort_month
    ORDER BY c.cohort_year, c.cohort_month
""").df()

print('=== Sample Cohort Retention (% with PASS call at month 0, 1, 2, 3) ===')
print(retention.head(12).to_string(index=False))